# Tutorial 1: Semantic Search and Evaluation - Choosing the Right Embedding Model

*Level: Intermediate*

---

You're building a search system over a corpus of scientific abstracts. Before you ship it, you need to answer two questions: does it actually find the right papers, and is the embedding model you picked the right one for the job? This tutorial walks through both. How to build the system with Qdrant, and how to know whether it's working.

## What We'll Build

A semantic search pipeline: documents in, vectors out, a Qdrant collection that can answer queries by meaning. We embed our corpus with three models from FastEmbed, run real queries against each, and measure how well each one retrieves the documents we actually want. Because "it returned something" isn't the same as "it returned the right thing."

## The Dataset: SciFact

We use SciFact, a scientific fact-checking dataset from [BEIR](https://huggingface.co/datasets/BeIR/scifact) (Benchmarking Information Retrieval), a standard evaluation suite for Information Retrieval systems. It assesses retrieval models by matching scientific claims to supporting or refuting evidence in biomedical texts.

What makes it suitable for our tutorial: it comes with ground-truth relevance judgments (qrels) that tell us, for each query, which documents are relevant. This allows us to measure retrieval quality objectively rather than eyeballing results.

To keep ingestion fast, we will only ingest the documents that appear in the qrels test split.

## What We'll Evaluate

For each embedding model we track how well the retrieved documents match the query, measured with standard IR metrics (NDCG@K, MRR, Recall@K, Precision@K). We'll also test whether the differences between models are statistically meaningful.

## What We'll Use

FastEmbed (Qdrant's lightweight embedding library, ONNX-optimized), Qdrant (our vector database), and ranx (a ranking evaluation library for NDCG, MRR, and significance testing against ground truth).

> Note: This notebook runs on CPU. Ingestion takes a few minutes. For full SciFact or faster iteration, switch to a T4 GPU runtime in Colab.

---

## 0. Setup

We start by installing the required libraries:

- **fastembed**: Qdrant's lightweight embedding library.
- **qdrant-client**: the Python client for Qdrant. It allows you to interact with your Qdrant cluster directly from Python.
- **datasets**: Hugging Face's library for loading datasets.
- **ranx**: a ranking evaluation library to compute retrieval metrics against ground truth.
- **tqdm**: progress bars for long-running loops.

In [ ]:
# GPU optional: if you switched to a GPU runtime, replace the fastembed install below
# with these two lines instead:
# !pip install onnxruntime-gpu -i https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/ -qqq
# !pip install fastembed-gpu==0.7.4 qdrant-client==1.16.1 datasets==4.5.0 ranx==0.3.21 tqdm==4.67.3 -qqq

!pip install fastembed==0.7.4 qdrant-client==1.16.1 datasets==4.5.0 ranx==0.3.21 tqdm==4.67.3 -qqq

In [ ]:
import time
import numpy as np
from tqdm import tqdm
from collections import defaultdict

from datasets import load_dataset
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from fastembed import TextEmbedding
from ranx import Qrels, Run, evaluate, compare

## 1. Create a Qdrant Cluster and Set Up a Client Connection

If you do not already have a Qdrant cluster, follow these steps to create one:

- Register for a [Qdrant Cloud account](https://cloud.qdrant.io/) using your email, Google, or Github credentials.
- Under Create a Free Cluster, enter a cluster name and select your preferred cloud provider and region.
- Click Create Free Cluster.
- Copy the API key when prompted and store it somewhere safe as it won't be displayed again.
- Copy the Cluster Endpoint. It should look something like `https://xxx.cloud.qdrant.io`.


Create a free cluster at [cloud.qdrant.io](https://cloud.qdrant.io).  
In Colab: open **Secrets** (key icon on the left sidebar) and add:
- `QDRANT_URL` : your cluster endpoint e.g. `https://xyz.us-east4-0.gcp.cloud.qdrant.io`
- `QDRANT_API_KEY` : your API key

Next create a client connection to your Qdrant cluster

In [ ]:
from google.colab import userdata

QDRANT_URL     = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

COLLECTION_NAME = "tutorial1_scifact-collection"
exists = client.collection_exists(COLLECTION_NAME)
print(f"Connected. '{COLLECTION_NAME}' exists: {exists}")

/tmp/ipykernel_20988/3052598536.py:6: UserWarning: Qdrant client version 1.16.1 is incompatible with server version 1.18.2. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)


Connected. 'tutorial1_scifact-collection' exists: True


---
## 2. Load SciFact

SciFact is fully public on Hugging Face. We load three splits:
- **corpus**: the biomedical texts we'll index
- **queries**: short scientific claims
- **qrels**: ground-truth relevance judgments

We concatenate `title + text` for each document and store them in a corpus dictionary keyed by document ID. Titles in scientific abstracts are typically informative and keyword-rich, so including them alongside the abstract generally improves retrieval quality.

To keep ingestion fast, we will only ingest the documents that appear in the qrels.

### Relevance Judgments (qrels)

Qrels (query relevance judgments) are the ground truth that makes evaluation possible. They map each query to its relevant documents with a relevance score of **1**. Any (query, document) pair absent from the qrels is treated as irrelevant by the evaluation framework.

We load the **test split**, which contains 300 queries.

In [ ]:
corpus_dataset = load_dataset("BeIR/scifact", "corpus", split="corpus")

doc_ids      = [str(doc["_id"]) for doc in corpus_dataset]
doc_titles   = [doc["title"]    for doc in corpus_dataset]
doc_texts    = [doc["text"]     for doc in corpus_dataset]
doc_passages = [(t + ". " + x).strip() if t else x for t, x in zip(doc_titles, doc_texts)]

print(f"Full corpus: {len(doc_ids)} documents")

queries_dataset = load_dataset("BeIR/scifact", "queries", split="queries")
queries = {str(q["_id"]): q["text"] for q in queries_dataset}

qrels_dataset = load_dataset("BeIR/scifact-qrels", split="test")
qrels_dict    = defaultdict(dict)
for row in qrels_dataset:
    qrels_dict[str(row["query-id"])][str(row["corpus-id"])] = row["score"]

eval_queries = {qid: queries[qid] for qid in qrels_dict if qid in queries}
qrels_ranx   = Qrels(dict(qrels_dict))

# Filter corpus to only documents referenced in qrels
# Ranx treats any document not in qrels for a given query as grade 0
# so evaluation remains fully valid with this filtered corpus
qrel_doc_ids = set(
    doc_id
    for doc_dict in qrels_dict.values()
    for doc_id in doc_dict.keys()
)

mask         = [did in qrel_doc_ids for did in doc_ids]
doc_ids      = [x for x, m in zip(doc_ids,      mask) if m]
doc_titles   = [x for x, m in zip(doc_titles,   mask) if m]
doc_texts    = [x for x, m in zip(doc_texts,    mask) if m]
doc_passages = [x for x, m in zip(doc_passages, mask) if m]

corpus = {doc_ids[i]: doc_passages[i] for i in range(len(doc_ids))}

# Stats
counts = [len(v) for v in qrels_dict.values()]
print(f"Filtered corpus : {len(doc_ids)} documents")
print(f"Evaluation queries : {len(eval_queries)}")
print(f"Relevant docs per query: mean={np.mean(counts):.1f}  min={min(counts)}  max={max(counts)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Full corpus: 5183 documents
Filtered corpus : 283 documents
Evaluation queries : 300
Relevant docs per query: mean=1.1  min=1  max=5


Document lengths and a sample query with its relevant documents below. This gives a concrete sense of what we're working with before jumping into embeddings.

In [ ]:
# Document length distribution
lengths = [len(t.split()) for t in doc_passages]
print(f"Doc length (words): max={np.max(lengths):.0f}  mean={np.mean(lengths):.0f}  median={np.median(lengths):.0f}  "
      f"p10={np.percentile(lengths,10):.0f}  p90={np.percentile(lengths,90):.0f}")

# Sample query with its relevant documents
ex_qid = list(eval_queries.keys())[5]
print(f"\nSample query: '{eval_queries[ex_qid]}'")
print("Top relevant docs (by grade):")
for doc_id, score in sorted(qrels_dict[ex_qid].items(), key=lambda x: -x[1])[:3]:
    print(f"  [grade={score}] {doc_id}: {corpus.get(doc_id,'')[:120]}...")

Doc length (words): max=1070  mean=230  median=189  p10=143  p90=371

Sample query: 'A high microerythrocyte count raises vulnerability to severe anemia in homozygous alpha (+)- thalassemia trait subjects.'
Top relevant docs (by grade):
  [grade=1] 18174210: Increased Microerythrocyte Count in Homozygous α+-Thalassaemia Contributes to Protection against Severe Malarial Anaemia...


---
## 3. Define the Embedding Models

Choosing the right embedding model is one of the most impactful decisions when building a retrieval pipeline. It directly affects the **quality** of the documents you retrieve and the **latency** of your system. In this tutorial we focus on quality.

To understand this impact, we compare three models:

- `bge-small-en-v1.5`: compact and fast, 384 dimensions
- `bge-base-en-v1.5`: stronger general-purpose model, 768 dimensions
- `nomic-embed-text-v1.5`: 768 dimensions, much larger context window

We deliberately mix model families. The two BGE models let us see whether more capacity within the same family translates to better retrieval, while `nomic-embed-text-v1.5` brings a different training distribution and an 8K context window for contrast.

> Nomic-embed-text-v1.5 is trained to expect task-specific prefixes (`search_document:` for passages, `search_query:` for queries). FastEmbed does not apply these prefixes automatically. To compare nomic fairly with the BGE models, we wire the prefix logic in ourselves below. Always check whether your chosen model has prefix requirements before trusting the numbers.

Feel free to experiment with other models. Run `TextEmbedding.list_supported_models()` to see everything available in FastEmbed.

In [ ]:
model_info = {m["model"]: m for m in TextEmbedding.list_supported_models()}

def get_context_window(model_name):
    desc = model_info.get(model_name, {}).get("description", "")
    import re
    match = re.search(r"(\d+)\s*input tokens", desc)
    return int(match.group(1)) if match else "unknown"

# List of models we'll experiment with in this tutorial, feel free to explore other models

# Use GPU if available, fall back to CPU if not
USE_GPU = False  # set to True if you installed fastembed-gpu and are on a GPU runtime
providers = ["CUDAExecutionProvider"] if USE_GPU else None

# Define our embedding models
bge_small = TextEmbedding(model_name="BAAI/bge-small-en-v1.5",         providers=providers)
bge_base  = TextEmbedding(model_name="BAAI/bge-base-en-v1.5",          providers=providers)
nomic     = TextEmbedding(model_name="nomic-ai/nomic-embed-text-v1.5", providers=providers)

# doc_prefix and query_prefix capture per-model prefix requirements that FastEmbed
# does not apply automatically. Nomic is trained on these tokens; BGE is not.
MODELS = [
    {"name": "BAAI/bge-small-en-v1.5",         "dim": 384, "vector_name": "bge-small", "instance": bge_small, "context_window": get_context_window("BAAI/bge-small-en-v1.5"),         "doc_prefix": None,                "query_prefix": None},
    {"name": "BAAI/bge-base-en-v1.5",          "dim": 768, "vector_name": "bge-base",  "instance": bge_base,  "context_window": get_context_window("BAAI/bge-base-en-v1.5"),          "doc_prefix": None,                "query_prefix": None},
    {"name": "nomic-ai/nomic-embed-text-v1.5", "dim": 768, "vector_name": "nomic",     "instance": nomic,     "context_window": get_context_window("nomic-ai/nomic-embed-text-v1.5"), "doc_prefix": "search_document: ", "query_prefix": "search_query: "},
]

def embed_with_prefix(model_instance, texts, prefix=None):
    """Prepend a model-specific prefix if one is provided, then embed.

    Use prefix=model['doc_prefix'] for passages, prefix=model['query_prefix'] for queries.
    Returns a list of NumPy arrays, one per input text.
    """
    if prefix:
        texts = [f"{prefix}{t}" for t in texts]
    return list(model_instance.embed(texts))

print("Models ready:")
for m in MODELS:
    pfx_note = " (prefix required)" if m["doc_prefix"] else ""
    print(f"  [{m['vector_name']:10s}] dim={m['dim']:4d}  context={m['context_window']} tokens  size={model_info.get(m['name'],{}).get('size_in_GB','?')}GB{pfx_note}")

Models ready:
  [bge-small ] dim= 384  context=512 tokens  size=0.067GB
  [bge-base  ] dim= 768  context=512 tokens  size=0.21GB
  [nomic     ] dim= 768  context=8192 tokens  size=0.52GB (prefix required)


### Embedding Models and Context Windows

Every embedding model has a **context window:** a maximum number of tokens it can process at once. Text that exceeds this limit gets **truncated** and is never represented in the embedding.

When building a retrieval system, you should chunk your documents to fit within your embedding model's context window. The tokenizer adds special tokens around your text, so a 512-token model can't encode 512 tokens of your content.

We'll check what percentage of our filtered SciFact documents exceed 512 tokens. For those that do, the model will silently truncate them.

> In production, chunk your documents to fit your model's context window. Truncation means information loss.

In [ ]:
# we'll use bge_small's tokenizer as an example
tokenizer = bge_small.model.tokenizer
# Disable truncation to measure true token lengths
tokenizer.no_truncation()

true_token_counts = [len(tokenizer.encode(t).ids) for t in doc_passages]

over_512 = [(doc_ids[i], true_token_counts[i])
            for i in range(len(doc_texts))
            if true_token_counts[i] > 512]

print(f"True token stats: mean={np.mean(true_token_counts):.0f}  "
      f"median={np.median(true_token_counts):.0f}  "
      f"p90={np.percentile(true_token_counts,90):.0f}  "
      f"max={max(true_token_counts)}")
print(f"\nDocs exceeding 512 tokens: {len(over_512)} ({len(over_512)/len(doc_texts)*100:.1f}%)")

# Re-enable truncation (models require sequences ≤ their context window)
tokenizer.enable_truncation(max_length=512)

True token stats: mean=366  median=314  p90=579  max=1364

Docs exceeding 512 tokens: 46 (16.3%)


In [ ]:
# Defensive check: verify what each model actually receives.
# Nomic requires "search_document: " for passages and "search_query: " for queries.
# FastEmbed does not apply these automatically, so we inspect the tokenized
# output to confirm the prefix is present where required.

sample = "vector databases store embeddings"

print("Without explicit prefix (raw model.embed input):")
for m in MODELS:
    tokens = m["instance"].model.tokenizer.encode(sample).tokens
    print(f"  [{m['vector_name']:10s}] first 8 tokens: {tokens[:8]}")

print("\nWith doc_prefix applied (what we'll send during ingestion):")
for m in MODELS:
    prefixed = f"{m['doc_prefix']}{sample}" if m["doc_prefix"] else sample
    tokens = m["instance"].model.tokenizer.encode(prefixed).tokens
    print(f"  [{m['vector_name']:10s}] first 8 tokens: {tokens[:8]}")

Without explicit prefix (raw model.embed input):
  [bge-small ] first 8 tokens: ['[CLS]', 'vector', 'databases', 'store', 'em', '##bed', '##ding', '##s']
  [bge-base  ] first 8 tokens: ['[CLS]', 'vector', 'databases', 'store', 'em', '##bed', '##ding', '##s']
  [nomic     ] first 8 tokens: ['[CLS]', 'vector', 'databases', 'store', 'em', '##bed', '##ding', '##s']

With doc_prefix applied (what we'll send during ingestion):
  [bge-small ] first 8 tokens: ['[CLS]', 'vector', 'databases', 'store', 'em', '##bed', '##ding', '##s']
  [bge-base  ] first 8 tokens: ['[CLS]', 'vector', 'databases', 'store', 'em', '##bed', '##ding', '##s']
  [nomic     ] first 8 tokens: ['[CLS]', 'search', '_', 'document', ':', 'vector', 'databases', 'store']


---
## 4. Create the Collection and Ingest the Data

Now it's time to create a Qdrant collection and ingest our documents into it. We create a collection with **named vectors** (one for each embedding). This way a single point with a single payload holds all three embeddings simultaneously.

Each point looks like this:

```python
PointStruct(
    id=0,
    vector={
        "bge-small": [...],  # 384-dim
        "bge-base":  [...],  # 768-dim
        "nomic":     [...],  # 768-dim
    },
    payload={
        "doc_id": "..."
    }
)
```

First, create the Qdrant collection.

In [ ]:
def create_collection(client, collection_name, models):
    if client.collection_exists(collection_name):
        client.delete_collection(collection_name)
        print(f"Deleted existing '{collection_name}'")

    vectors_config = {
        m["vector_name"]: VectorParams(size=m["dim"], distance=Distance.COSINE)
        for m in models
    }

    client.create_collection(
        collection_name=collection_name,
        vectors_config=vectors_config,
    )
    print(f"Created '{collection_name}' with named vectors:")
    for m in models:
        print(f"  '{m['vector_name']}' , {m['dim']}d cosine")

create_collection(client, COLLECTION_NAME, MODELS)

Deleted existing 'tutorial1_scifact-collection'
Created 'tutorial1_scifact-collection' with named vectors:
  'bge-small' , 384d cosine
  'bge-base' , 768d cosine
  'nomic' , 768d cosine


Define the data ingestion function.

In [ ]:
def ingest_corpus(collection_name, models, corpus, batch_size=16):
    doc_ids   = list(corpus.keys())
    passages  = list(corpus.values())
    total     = len(doc_ids)

    # Resume cursor: if the collection already has points, pick up where we left off.
    # This assumes points were written densely from id=0 in this corpus's order ,
    # convenient for tutorial re-runs, but fragile in production. For real workloads,
    # use explicit doc_id tracking or wipe and re-ingest rather than relying on count.
    start_idx = client.count(collection_name).count

    if start_idx > 0:
        print(f"Resuming from index {start_idx}")

    t0 = time.time()
    for batch_start in tqdm(range(start_idx, total, batch_size), desc="Ingesting"):
        batch_end     = min(batch_start + batch_size, total)
        batch_ids     = doc_ids[batch_start:batch_end]
        batch_passages = passages[batch_start:batch_end]

        # Embed with all models, applying per-model doc_prefix when required
        batch_vectors = {
            m["vector_name"]: [
                e.tolist()
                for e in embed_with_prefix(m["instance"], batch_passages, m.get("doc_prefix"))
            ]
            for m in models
        }
        # Batch upsert to the collection.
        # wait=True blocks until points are committed and queryable , without it,
        # client.upsert is fire-and-forget on Qdrant Cloud, and the eval cells
        # that follow can race ahead of indexing.
        client.upsert(
            collection_name=collection_name,
            points=[
                PointStruct(
                    id=batch_start + i,
                    vector={vn: batch_vectors[vn][i] for vn in batch_vectors},
                    payload={"doc_id": batch_ids[i]},
                )
                for i in range(len(batch_passages))
            ],
            wait=True,
        )

    print(f"Done in {time.time()-t0:.1f}s. {total} points in '{collection_name}'.")

Ingest the data into the collection.

In [ ]:
ingest_corpus(
    collection_name=COLLECTION_NAME,
    models=MODELS,
    corpus=corpus
)

Ingesting: 100%|██████████| 36/36 [32:34<00:00, 54.29s/it]

Done in 1954.3s. 283 points in 'tutorial1_scifact-collection'.


At this point you have a working semantic search system. Each document has been turned into three vector representations (one per model) and stored in Qdrant, along with the document text in its payload. To search, we embed a query with the same model, ask Qdrant for the nearest vectors, and read back the document text.

Before we measure how well it works, we run it. Pick a query, retrieve the top results, look at what comes back.


In [ ]:
# Pick any query from the dataset and search with bge-base
demo_query = "How do mutations in the BRCA1 gene affect breast cancer risk?"
demo_model = next(m for m in MODELS if m["vector_name"] == "bge-base")

qvec = embed_with_prefix(demo_model["instance"], [demo_query], demo_model.get("query_prefix"))[0].tolist()

hits = client.query_points(
    collection_name=COLLECTION_NAME,
    query=qvec,
    using=demo_model["vector_name"],
    limit=3,
    with_payload=True,
).points

print(f"Query: {demo_query}\n")

# Verify top hit is BRCA1-relevant before proceeding
top_hit = hits[0]
print(f"Top hit content verification: {corpus[top_hit.payload['doc_id']][:200]}...")
print()

for rank, hit in enumerate(hits, 1):
    doc_text = corpus[hit.payload["doc_id"]]
    print(f"#{rank} (score={hit.score:.4f}) doc_id={hit.payload['doc_id']}")
    print(f"   {doc_text[:200]}...\n")

Query: How do mutations in the BRCA1 gene affect breast cancer risk?

Top hit content verification: Gene–environment interactions in 7610 women with breast cancer: prospective evidence from the Million Women Study. BACKGROUND Information is scarce about the combined effects on breast cancer incidenc...

#1 (score=0.7601) doc_id=18340282
   Gene–environment interactions in 7610 women with breast cancer: prospective evidence from the Million Women Study. BACKGROUND Information is scarce about the combined effects on breast cancer incidenc...

#2 (score=0.7465) doc_id=13519661
   Linkage Disequilibrium Mapping of       CHEK2: Common Variation and Breast Cancer Risk     . Background Checkpoint kinase 2 (CHEK2) averts cancer development by promoting cell cycle arrest and activat...

#3 (score=0.7436) doc_id=4414547
   Mosaic PPM1D mutations are associated with predisposition to breast and ovarian cancer. Improved sequencing technologies offer unprecedented opportunities for investigating th

That's semantic search: a query in natural language, ranked results by meaning. No keyword overlap required. The top hit doesn't need to contain "BRCA1" verbatim; it just needs to be about it in vector space.

But "looks right" isn't an answer. The query above happens to return plausible results; another query might quietly return something subtly off. And with three embedding models loaded, we don't know which one we should actually be using. That's what the rest of the tutorial is for: turning *"the search returned something"* into *"the search returned the right thing, reliably, and here's how we know."*


---
## 5. Evaluate Retrieval Quality

We'll use standard IR metrics to measure how well each model retrieves the right documents:

### NDCG@k ([Normalized Discounted Cumulative Gain](https://en.wikipedia.org/wiki/Discounted_cumulative_gain))

NDCG@k rewards placing highly relevant documents early in the ranking. It discounts by position (rank 1 matters more than rank 10) and normalizes by the best possible score for that query. The result is between 0 and 1.

### MRR ([Mean Reciprocal Rank](https://en.wikipedia.org/wiki/Mean_reciprocal_rank))

MRR averages 1/rank of the first relevant result across all queries. Used when you expect one correct answer per query and care about how early it appears.

In [ ]:
def evaluate_model(collection_name, model_entry,
                   eval_queries, qrels_ranx, top_k=5):
    """
    For each evaluation query:
    Collect NDCG@5, MRR, Precision@5, Recall@5
    """
    model        = model_entry["instance"]
    vector_name  = model_entry["vector_name"]
    query_prefix = model_entry.get("query_prefix")

    results = {}   # qid -> {doc_id: score} : ranx input format

    for qid, qtext in tqdm(eval_queries.items(), desc=f"  [{vector_name}]"):

        qvec = embed_with_prefix(model, [qtext], query_prefix)[0].tolist()

        hits = client.query_points(
            collection_name=collection_name,
            query=qvec,
            using=vector_name,
            limit=top_k,
            with_payload=True,
        ).points

        results[qid] = {hit.payload["doc_id"]: hit.score for hit in hits}

    # Compute retrieval metrics against ground truth
    metrics = evaluate(
        qrels_ranx,
        Run(results),
        [f"ndcg@{top_k}", f"precision@{top_k}", f"recall@{top_k}", "mrr"]
    )

    return {
        "ndcg@5":      metrics[f"ndcg@{top_k}"],
        "mrr":          metrics["mrr"],
        "precision@5": metrics[f"precision@{top_k}"],
        "recall@5":    metrics[f"recall@{top_k}"]
    }

Run the evaluation.

In [ ]:
eval_results = {}

for m in MODELS:
    print(f"\nEvaluating [{m['vector_name']}]")
    metrics = evaluate_model(
        COLLECTION_NAME,
        m,
        eval_queries, qrels_ranx, top_k=5
    )
    eval_results[m["vector_name"]] = metrics
    print(f"  NDCG@5={metrics['ndcg@5']:.4f}  "
          f"Precision@5={metrics['precision@5']:.4f}  "
          f"Recall@5={metrics['recall@5']:.4f}  "
          f"MRR={metrics['mrr']:.4f}")


Evaluating [bge-small]


  [bge-small]: 100%|██████████| 300/300 [00:59<00:00,  5.05it/s]
/usr/local/lib/python3.12/dist-packages/ranx/metrics/ndcg.py:72: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  scores[i] = _ndcg(qrels[i], run[i], k, rel_lvl, jarvelin)


  NDCG@5=0.8925  Precision@5=0.2113  Recall@5=0.9343  MRR=0.8809

Evaluating [bge-base]


  [bge-base]: 100%|██████████| 300/300 [01:25<00:00,  3.52it/s]


  NDCG@5=0.9028  Precision@5=0.2147  Recall@5=0.9503  MRR=0.8886

Evaluating [nomic]


  [nomic]: 100%|██████████| 300/300 [01:40<00:00,  2.99it/s]

  NDCG@5=0.8754  Precision@5=0.2093  Recall@5=0.9253  MRR=0.8605


---
## 6. Results

In [ ]:
rows = []
for m in MODELS:
    vn = m["vector_name"]
    r  = eval_results[vn]
    rows.append({
        "Model":        vn,
        "Dim":          m["dim"],
        "NDCG@5":       round(r["ndcg@5"], 4),
        "Precision@5":  round(r["precision@5"], 4),
        "Recall@5":     round(r["recall@5"], 4),
        "MRR":          round(r["mrr"], 4),
    })

rows.sort(key=lambda x: x["NDCG@5"], reverse=True)

# Print as aligned table
header = list(rows[0].keys())
col_w  = {h: max(len(h), max(len(str(r[h])) for r in rows)) for h in header}
fmt    = "  ".join(f"{{:<{col_w[h]}}}" for h in header)
print(fmt.format(*header))
print("  ".join("-" * col_w[h] for h in header))
for row in rows:
    print(fmt.format(*[str(row[h]) for h in header]))

Model      Dim  NDCG@5  Precision@5  Recall@5  MRR   
---------  ---  ------  -----------  --------  ------
bge-base   768  0.9028  0.2147       0.9503    0.8886
bge-small  384  0.8925  0.2113       0.9343    0.8809
nomic      768  0.8754  0.2093       0.9253    0.8605


**Three models, about three NDCG@5 points apart; only the `bge-base` vs `nomic` gap clears the significance bar.**

The three models span about 2.7 points of NDCG@5: `bge-base` on top, `bge-small` close behind, and `nomic` noticeably lower. Before reading this as a clean ranking, we should ask whether each pairwise gap is statistically meaningful given the size of our evaluation set. With 300 queries and an average of 1.1 relevant documents each, a 1-point gap might plausibly flip on a different sample of queries. A 2.7-point gap is harder to dismiss but still worth testing.

We test that directly below.


In [ ]:
runs = []
for m in MODELS:
    vn = m["vector_name"]
    run_dict = {}
    for qid, qtext in eval_queries.items():
        qvec = embed_with_prefix(m["instance"], [qtext], m.get("query_prefix"))[0].tolist()
        hits = client.query_points(
            collection_name=COLLECTION_NAME,
            query=qvec,
            using=vn,
            limit=5,
            with_payload=True,
        ).points
        run_dict[qid] = {hit.payload["doc_id"]: hit.score for hit in hits}
    run = Run(run_dict)
    run.name = vn
    runs.append(run)

report = compare(
    qrels=qrels_ranx,
    runs=runs,
    metrics=["ndcg@5", "mrr"],
    max_p=0.05,
    stat_test="fisher",
)
print(report)

#    Model      NDCG@5    MRR
---  ---------  --------  ------
a    bge-small  0.892     0.881
b    bge-base   0.903ᶜ    0.889ᶜ
c    nomic      0.875     0.860


Ranx runs a paired Fisher randomization test by default and marks significant differences at p < 0.05. Each row gets a letter (a, b, c); a superscript on a score means that model significantly outperforms the model identified by that letter. So `bge-base`'s `0.903ᶜ` reads: `bge-base` significantly outperforms model c (`nomic`). On this dataset, that's the only significant pairwise difference: the BGE pair is statistically tied, and `bge-small` vs `nomic` doesn't clear the bar.

When two candidates are tied on quality, the decision shifts to other axes: latency, memory, sequence length, cost. `bge-small` (384 dimensions, ~67MB) becomes the obvious default unless something else changes the picture. Read quality numbers as a screening filter, not a final ranking.

### What This SciFact Evaluation Teaches About Building Your Own

SciFact's professionally annotated qrels represent the gold standard: 300 queries with expert relevance judgments, enough statistical power to detect real differences between models. Most teams start with nothing: no ground truth, no labels, just a corpus and a vague sense of what users will ask.

The minimum viable starting point is smaller than you think: 10-20 hand-written queries, each tagged with the documents you judge relevant. But phrase queries the way *users* will phrase them, not drawn from document text. Reverse-engineered queries (writing questions from answers) saturate at perfect scores and teach you nothing about model differences.

For real signal, hand-tag relevant docs by skimming top-K retrieval from a baseline model (faster than reading the full corpus). Grow the set organically: the queries your best model fails on are the most valuable to add next. For directional comparison, 50-100 queries often suffices; for the statistical rigor we demonstrated above, you typically need a few hundred.

The same metrics for an example query:

In [ ]:
sample_qid  = list(eval_queries.keys())[100]
sample_text = eval_queries[sample_qid]
all_relevant = [doc_id for doc_id, g in qrels_dict.get(sample_qid, {}).items() if g >= 1]

print(f"Query: '{sample_text}'")
print(f"Total relevant docs: {len(all_relevant)}")
print()

for m in MODELS:
    qvec = embed_with_prefix(m["instance"], [sample_text], m.get("query_prefix"))[0].tolist()
    hits = client.query_points(
        collection_name=COLLECTION_NAME,
        query=qvec,
        using=m["vector_name"],
        limit=5,
        with_payload=True,
    ).points

    retrieved_ids = [hit.payload["doc_id"] for hit in hits]
    grades        = [qrels_dict.get(sample_qid, {}).get(doc_id, 0) for doc_id in retrieved_ids]

    single_qrels = Qrels({sample_qid: qrels_dict[sample_qid]})
    single_run   = Run({sample_qid: {doc_id: hit.score for doc_id, hit in zip(retrieved_ids, hits)}})
    metrics      = evaluate(single_qrels, single_run, ["ndcg@5", "precision@5", "recall@5", "mrr"])

    print(f"[{m['vector_name']}]")
    for rank, (doc_id, grade) in enumerate(zip(retrieved_ids, grades), 1):
        print(f"  #{rank} [grade={grade}] {doc_id}")
    print(f"  NDCG@5={metrics['ndcg@5']:.4f}  Precision@5={metrics['precision@5']:.4f}  "
          f"Recall@5={metrics['recall@5']:.4f}  MRR={metrics['mrr']:.4f}")
    print()

### What These Numbers Don't Tell You

Three caveats before you generalize from this table. First, the 283-document filter compresses scores: `bge-base` scores 0.7434 NDCG@10 on the full 5,183-doc SciFact corpus (from the [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard)), not 0.90. Expect inter-model spread to widen at full scale. Second, we measured only quality. Throughput, latency, memory, sequence length, and cost often differ by 5x or more between models tied on quality (see [How to Choose an Embedding Model](https://qdrant.tech/articles/how-to-choose-an-embedding-model/) for the full framework). Third, none of this was on your data. The methodology transfers; the numbers don't.

The next tutorial in this series (Tutorial 2: Hybrid Search) builds on this evaluation harness to compare dense, sparse, and hybrid retrieval against the same ground truth.

---
## 7. Takeaways and closing thoughts

**Embedding model choice matters:**

There is no universally best model. Factors like training distribution and optimization objective might have a bigger impact than raw size. Always evaluate on your actual data before committing to a model. In a more advanced step, consider finetuning your embedding model on your own data.

**Evaluation is a continuous process:**

The scores you saw here are a starting point, not a finish line. Start by evaluating on a dataset close to your domain (ideally one with human-annotated relevance judgments). As your data grows, your query distribution shifts, and your users' needs evolve, retrieval quality should be re-evaluated regularly. Over time, incorporate signals from real usage: click-through rates, user ratings, session abandonment... These tell you how retrieval quality translates into actual user experience.

**Retrieval quality depends on more than embeddings:**

Our toy dataset of 283 documents stayed far below Qdrant's default HNSW indexing threshold of 10,000 kB, which means Qdrant ran **exact nearest-neighbor search** (full KNN brute-force) for all queries. The scores you see here reflect pure embedding quality. In production, with larger collections of hundreds of millions or even billions of points, you can no longer rely on exact brute-force search, as computing distances to every vector becomes prohibitively expensive in both latency and cost. Instead, **approximate nearest-neighbor (ANN)** algorithms such as HNSW are used to trade a small amount of recall for massive gains in speed and scalability.
To learn how to tune HNSW for retrieval quality, see the [Qdrant retrieval quality tutorial](https://qdrant.tech/documentation/tutorials-search-engineering/retrieval-quality/).

**The quality-latency-memory tradeoff:**

In this tutorial we focused exclusively on retrieval quality. Latency was not measured because at this scale (a few thousand documents on a cloud instance), search time is dominated by network round-trip rather than actual computation, making the numbers uninformative.

In production, latency and memory matter just as much, and both scale with corpus size and vector dimension. For latency, the metrics worth tracking are **P50** (median, your typical user experience) and **P99** (tail latency, your worst 1% of requests). For memory, a higher-dimensional model stores more floats per vector, which adds up quickly across millions of documents. **Quantization** (storing vectors at lower precision, for example int8 instead of float32) cuts memory substantially with minimal quality loss, and is often what makes a larger, higher-quality model practical at scale.